In [ ]:
# =====================================
# 1. Mount Google Drive
# =====================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


# =====================================
# 2. Imports
# =====================================
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score


# =====================================
# 3. Load Dataset
# =====================================
path = "/content/drive/MyDrive/colab_datasets/Indo-HateSpeech_Dataset.xlsx"
df = pd.read_excel(path)

# Clean column names
df.columns = df.columns.str.strip()

print("Columns in dataset:", df.columns)
print("Total rows before cleaning:", len(df))


# =====================================
# 4. Keep Required Columns
# =====================================
df = df[['Comment', 'Label']]
df.dropna(inplace=True)

print("Rows after dropping NA:", len(df))


# =====================================
# 5. Fix & Normalize Labels
# =====================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

# Remove unwanted single quotes:  'HS0' → HS0
df['Label'] = df['Label'].str.replace("'", "", regex=False)

print("Unique labels after fixing:", df['Label'].unique())


# =====================================
# 6. Binary Mapping
# HS0 → 0
# HS1 + HSN → 1
# =====================================
def map_label(x):
    if x == 'HS0':
        return 0
    elif x in ['HS1', 'HSN']:
        return 1
    else:
        return None

df['Label'] = df['Label'].apply(map_label)

# Remove invalid rows
df = df[df['Label'].notnull()]
df['Label'] = df['Label'].astype(int)

print("Rows after label mapping:", len(df))
print("Label distribution:\n", df['Label'].value_counts())


# =====================================
# 7. Clean Text
# =====================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['Comment'] = df['Comment'].apply(clean_text)


# =====================================
# 8. Train-Test Split
# =====================================
X_train, X_test, y_train, y_test = train_test_split(
    df['Comment'],
    df['Label'],
    test_size=0.2,
    random_state=42,
    stratify=df['Label']
)


# =====================================
# 9. TF-IDF Vectorization
# =====================================
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# =====================================
# 10. Train SVM Model
# =====================================
model = LinearSVC(class_weight='balanced')
model.fit(X_train_vec, y_train)


# =====================================
# 11. Evaluation
# =====================================
y_pred = model.predict(X_test_vec)

print("\n===== RESULTS =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Mounted at /content/drive
Columns in dataset: Index(['Source.Name', 'Column1', 'Column2', 'Date', 'Likes', 'Comment',
       '(view source)', 'Post ID', 'Label'],
      dtype='object')
Total rows before cleaning: 77926
Rows after dropping NA: 76566
Unique labels after fixing: ['HS0' 'HS1' 'HSN']
Rows after label mapping: 76566
Label distribution:
 Label
0    62834
1    13732
Name: count, dtype: int64

===== RESULTS =====
Accuracy: 0.9231422228026642

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.94      0.95     12567
           1       0.75      0.86      0.80      2747

    accuracy                           0.92     15314
   macro avg       0.86      0.90      0.88     15314
weighted avg       0.93      0.92      0.93     15314



In [ ]:
# =====================================
# 1. Mount Drive
# =====================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


# =====================================
# 2. Imports
# =====================================
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from transformers import EarlyStoppingCallback

# =====================================
# 3. Load Dataset
# =====================================
path = "/content/drive/MyDrive/colab_datasets/Indo-HateSpeech_Dataset.xlsx"
df = pd.read_excel(path)

df.columns = df.columns.str.strip()
df = df[['Comment', 'Label']]
df.dropna(inplace=True)

# Fix label formatting
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].str.replace("'", "", regex=False)

# Binary mapping
def map_label(x):
    if x == 'HS0':
        return 0
    elif x in ['HS1', 'HSN']:
        return 1
    else:
        return None

df['Label'] = df['Label'].apply(map_label)
df = df[df['Label'].notnull()]
df['Label'] = df['Label'].astype(int)

print("Final dataset size:", len(df))
print("Label distribution:\n", df['Label'].value_counts())


# =====================================
# 4. Train-Test Split
# =====================================
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Comment'],
    df['Label'],
    test_size=0.2,
    random_state=42,
    stratify=df['Label']
)


# Convert to HuggingFace Dataset
train_dataset = Dataset.from_dict({
    "text": list(train_texts),
    "label": list(train_labels)
})

test_dataset = Dataset.from_dict({
    "text": list(test_texts),
    "label": list(test_labels)
})


# =====================================
# 5. Load XLM-RoBERTa
# =====================================
model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


# =====================================
# 6. Metrics
# =====================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }


# =====================================
# 7. Training Arguments
# =====================================
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available()
)


# =====================================
# 8. Trainer
# =====================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)


# =====================================
# 9. Train
# =====================================
trainer.train()


# =====================================
# 10. Evaluate
# =====================================
results = trainer.evaluate()
print("\nFinal Results:", results)


Mounted at /content/drive
Final dataset size: 76566
Label distribution:
 Label
0    62834
1    13732
Name: count, dtype: int64


Map:   0%|          | 0/61252 [00:00<?, ? examples/s]

Map:   0%|          | 0/15314 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.148105,0.108104,0.965848,0.905133
2,0.095901,0.113246,0.975186,0.931284


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Results: {'eval_loss': 0.10833408683538437, 'eval_accuracy': 0.9659135431631187, 'eval_f1': 0.9053318824809575, 'eval_runtime': 29.2895, 'eval_samples_per_second': 522.849, 'eval_steps_per_second': 32.708, 'epoch': 2.0}


In [ ]:
# =====================================
# 1. Mount Drive
# =====================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# =====================================
# 2. Imports
# =====================================
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# =====================================
# 3. Load Dataset
# =====================================
path = "/content/drive/MyDrive/colab_datasets/Indo-HateSpeech_Dataset.xlsx"
df = pd.read_excel(path)

df.columns = df.columns.str.strip()
df = df[['Comment', 'Label']]
df.dropna(inplace=True)

df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].str.replace("'", "", regex=False)

def map_label(x):
    if x == 'HS0':
        return 0
    elif x in ['HS1', 'HSN']:
        return 1
    else:
        return None

df['Label'] = df['Label'].apply(map_label)
df = df[df['Label'].notnull()]
df['Label'] = df['Label'].astype(int)

print("Final dataset size:", len(df))
print("Label distribution:\n", df['Label'].value_counts())

# =====================================
# 4. Train / Dev / Test Split (80:10:10)
# =====================================

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['Comment'],
    df['Label'],
    test_size=0.2,
    random_state=42,
    stratify=df['Label']
)

dev_texts, test_texts, dev_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print("Train:", len(train_texts))
print("Dev:", len(dev_texts))
print("Test:", len(test_texts))

# =====================================
# 5. Convert to HuggingFace Dataset
# =====================================

train_dataset = Dataset.from_dict({
    "text": list(train_texts),
    "label": list(train_labels)
})

dev_dataset = Dataset.from_dict({
    "text": list(dev_texts),
    "label": list(dev_labels)
})

test_dataset = Dataset.from_dict({
    "text": list(test_texts),
    "label": list(test_labels)
})

# =====================================
# 6. Tokenization
# =====================================

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
dev_dataset = dev_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
dev_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])
test_dataset.set_format("torch", columns=["input_ids","attention_mask","label"])

# =====================================
# 7. Model
# =====================================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# =====================================
# 8. Metrics
# =====================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# =====================================
# 9. Training Arguments
# =====================================

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available()
)

# =====================================
# 10. Trainer
# =====================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# =====================================
# 11. Train
# =====================================

trainer.train()

# =====================================
# 12. Final Test Evaluation
# =====================================

results = trainer.evaluate(test_dataset)
print("\nFinal Test Results:", results)


Mounted at /content/drive
Final dataset size: 76566
Label distribution:
 Label
0    62834
1    13732
Name: count, dtype: int64
Train: 61252
Dev: 7657
Test: 7657


Map:   0%|          | 0/61252 [00:00<?, ? examples/s]

Map:   0%|          | 0/7657 [00:00<?, ? examples/s]

Map:   0%|          | 0/7657 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.302478,0.113902,0.957816,0.886865
2,0.099630,0.081058,0.974664,0.930366
3,0.064825,0.077539,0.975970,0.934798
4,0.047566,0.083081,0.980671,0.946103
5,0.037448,0.099665,0.975317,0.933568


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Test Results: {'eval_loss': 0.07770334184169769, 'eval_accuracy': 0.9754473031213269, 'eval_f1': 0.9324227174694465, 'eval_runtime': 12.7783, 'eval_samples_per_second': 599.219, 'eval_steps_per_second': 9.391, 'epoch': 5.0}


In [ ]:
# =====================================
# 1. Install & Mount
# =====================================
!pip install -q transformers datasets torch openpyxl

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# =====================================
# 2. Imports
# =====================================
import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# =====================================
# 3. Load Dataset
# =====================================
path = "/content/drive/MyDrive/colab_datasets/Indo-HateSpeech_Dataset.xlsx"
df = pd.read_excel(path)

df.columns = df.columns.str.strip()

print("Total rows:", len(df))

# =====================================
# 4. Keep Required Columns
# =====================================
df = df[['Comment', 'Label']]
df.dropna(inplace=True)

# =====================================
# 5. Fix Labels
# =====================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].str.replace("'", "", regex=False)

def map_label(x):
    if x == 'HS0':
        return 0
    elif x in ['HS1', 'HSN']:
        return 1
    else:
        return None

df['Label'] = df['Label'].apply(map_label)
df = df[df['Label'].notnull()]
df['Label'] = df['Label'].astype(int)

print("Label distribution:\n", df['Label'].value_counts())

# =====================================
# 6. Clean Text
# =====================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['Comment'] = df['Comment'].apply(clean_text)

# =====================================
# 7. Split (80:10:10)
# =====================================
X_train, X_temp, y_train, y_temp = train_test_split(
    df['Comment'], df['Label'],
    test_size=0.2,
    stratify=df['Label'],
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

# =====================================
# 8. Convert to Dataset
# =====================================
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df   = pd.DataFrame({'text': X_val, 'label': y_val})
test_df  = pd.DataFrame({'text': X_test, 'label': y_test})

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)

# =====================================
# 9. Tokenizer
# =====================================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# =====================================
# 10. Model
# =====================================
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# =====================================
# 11. Training Arguments
# =====================================
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    load_best_model_at_end=True
)

# =====================================
# 12. Metrics
# =====================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# =====================================
# 13. Trainer
# =====================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =====================================
# 14. Train
# =====================================
trainer.train()

# =====================================
# 15. Train Accuracy
# =====================================
train_output = trainer.predict(train_dataset)
train_preds = torch.argmax(torch.tensor(train_output.predictions), axis=1)

print("\n===== TRAIN RESULTS =====")
print("Train Accuracy:", accuracy_score(y_train, train_preds))
print("Train F1:", f1_score(y_train, train_preds))
print(classification_report(y_train, train_preds))

# =====================================
# 16. Test Accuracy
# =====================================
test_output = trainer.predict(test_dataset)
test_preds = torch.argmax(torch.tensor(test_output.predictions), axis=1)

print("\n===== TEST RESULTS =====")
print("Test Accuracy:", accuracy_score(y_test, test_preds))
print("Test F1:", f1_score(y_test, test_preds))
print(classification_report(y_test, test_preds))

Mounted at /content/drive
Total rows: 77926
Label distribution:
 Label
0    62834
1    13732
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.164794,0.181922,0.948544,0.860679
2,0.150712,0.151978,0.949850,0.867495
3,0.138255,0.161316,0.953376,0.877446
4,0.132542,0.161612,0.951678,0.873027
5,0.125027,0.187948,0.948413,0.867048


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]